In [ ]:
# Import NumPy for mathematical calculations and arrays.
import numpy as np

# Import Matplotlib for displaying the learned probability distribution.
import matplotlib.pyplot as plt


# Set a random seed so that results can be reproduced.
np.random.seed(42)


# Create a small binary dataset containing only 0 and 1.
data = np.array([
    [1, 1, 0],
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1]
])


# Store the number of visible units in the dataset.
n_visible = data.shape[1]


# Store the number of hidden units in the model.
n_hidden = 2


# Initialize the weights connecting visible and hidden units.
weights = np.random.normal(
    0,
    0.1,
    (n_visible, n_hidden)
)


# Initialize the visible unit biases with zeros.
visible_bias = np.zeros(n_visible)


# Initialize the hidden unit biases with zeros.
hidden_bias = np.zeros(n_hidden)


# Define the learning rate for updating model parameters.
learning_rate = 0.1


# Define the number of training iterations.
epochs = 1000


# Define a function to calculate the sigmoid probability.
def sigmoid(x):

    # Convert the input value into a probability between 0 and 1.
    return 1 / (1 + np.exp(-x))


# Define a function to sample binary values from probabilities.
def sample_binary(probabilities):

    # Generate random values between 0 and 1.
    random_values = np.random.rand(*probabilities.shape)

    # Return 1 when the random value is smaller than the probability.
    # Otherwise, return 0.
    return (random_values < probabilities).astype(int)


# Define a function to calculate hidden unit probabilities.
def hidden_probability(visible):

    # Calculate the weighted input received by hidden units.
    activation = np.dot(visible, weights) + hidden_bias

    # Convert the activation into probabilities using sigmoid.
    return sigmoid(activation)


# Define a function to calculate visible unit probabilities.
def visible_probability(hidden):

    # Calculate the weighted input received by visible units.
    activation = np.dot(hidden, weights.T) + visible_bias

    # Convert the activation into probabilities using sigmoid.
    return sigmoid(activation)


# Start the training process for the Boltzmann Machine.
for epoch in range(epochs):

    # Calculate hidden probabilities using the original data.
    positive_hidden_prob = hidden_probability(data)

    # Calculate the positive association between visible and hidden units.
    positive_association = np.dot(
        data.T,
        positive_hidden_prob
    )

    # Sample binary hidden states from the hidden probabilities.
    hidden_sample = sample_binary(positive_hidden_prob)

    # Reconstruct visible unit probabilities from the hidden sample.
    negative_visible_prob = visible_probability(hidden_sample)

    # Sample binary visible states from the reconstructed probabilities.
    negative_visible_sample = sample_binary(negative_visible_prob)

    # Calculate hidden probabilities for the reconstructed visible data.
    negative_hidden_prob = hidden_probability(
        negative_visible_sample
    )

    # Calculate the negative association between visible and hidden units.
    negative_association = np.dot(
        negative_visible_sample.T,
        negative_hidden_prob
    )

    # Calculate the difference between positive and negative associations.
    weight_update = (
        positive_association - negative_association
    ) / len(data)

    # Update the weights using the learning rate.
    weights += learning_rate * weight_update

    # Calculate the difference between original and reconstructed visible data.
    visible_update = np.mean(
        data - negative_visible_sample,
        axis=0
    )

    # Update the visible biases using the learning rate.
    visible_bias += learning_rate * visible_update

    # Calculate the difference between positive and negative hidden probabilities.
    hidden_update = np.mean(
        positive_hidden_prob - negative_hidden_prob,
        axis=0
    )

    # Update the hidden biases using the learning rate.
    hidden_bias += learning_rate * hidden_update

    # Display training progress every 100 epochs.
    if (epoch + 1) % 100 == 0:

        # Print the current epoch number.
        print("Epoch:", epoch + 1)


# Print the final learned weights.
print("\nLearned Weights:")
print(weights)


# Print the final visible biases.
print("\nVisible Bias:")
print(visible_bias)


# Print the final hidden biases.
print("\nHidden Bias:")
print(hidden_bias)


# Create a list to store generated visible samples.
generated_samples = []


# Generate 1000 samples using the trained model.
for i in range(1000):

    # Initialize the visible units randomly as binary values.
    visible = np.random.randint(0, 2, n_visible)

    # Perform Gibbs sampling for 20 steps.
    for step in range(20):

        # Calculate hidden unit probabilities from visible units.
        hidden_prob = hidden_probability(visible)

        # Sample hidden units from their probabilities.
        hidden = sample_binary(hidden_prob)

        # Calculate visible unit probabilities from hidden units.
        visible_prob = visible_probability(hidden)

        # Sample new visible units from the visible probabilities.
        visible = sample_binary(visible_prob)

    # Store the final visible sample.
    generated_samples.append(visible)


# Convert the generated samples into a NumPy array.
generated_samples = np.array(generated_samples)


# Find the unique binary patterns and their occurrence counts.
unique_samples, counts = np.unique(
    generated_samples,
    axis=0,
    return_counts=True
)


# Calculate the probability of each unique pattern.
probabilities = counts / len(generated_samples)


# Convert binary patterns into readable strings.
labels = [
    ''.join(map(str, sample))
    for sample in unique_samples
]


# Display each binary pattern and its estimated probability.
print("\nLearned Probability Distribution:")

# Loop through all unique patterns and probabilities.
for label, probability in zip(labels, probabilities):

    # Print the binary pattern and its probability.
    print(label, ":", round(probability, 4))


# Create a bar chart to visualize the learned distribution.
plt.bar(labels, probabilities)


# Add a title to the chart.
plt.title("Boltzmann Machine Probability Distribution")


# Label the horizontal axis.
plt.xlabel("Binary Patterns")


# Label the vertical axis.
plt.ylabel("Estimated Probability")


# Display the chart.
plt.show()